Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 1
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC # Resolve Hudi source ambiguity (UniForm vs Apache Hudi)
# MAGIC spark.conf.set("spark.databricks.delta.uniform.hudi.enabled", "false"); spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
# MAGIC
# MAGIC
# MAGIC
# MAGIC spark.sql.catalogImplementation = "hive"
# MAGIC spark.sql.extensions = "org.apache.spark.sql.hudi.HoodieSparkSessionExtension"
# MAGIC spark.sql.catalog.hudi = "org.apache.spark.sql.hudi.catalog.HoodieCatalog"
# MAGIC spark.conf.set("spark.sql.hudi.datasource.read.allow-full-scan", "true")
# MAGIC spark.conf.set("spark.sql.hudi.datasource.write.allow-full-scan", "true")
# MAGIC spark.conf.set("spark.sql.hudi.datasource.read.source", "org.apache.hudi")
# MAGIC
# MAGIC
# MAGIC query = """
# MAGIC WITH parameters AS (
# MAGIC     SELECT
# MAGIC         CASE
# MAGIC             WHEN day_of_week(current_date) = 1 THEN date_add('day', -2, current_date)  -- Monday → include last 2 days
# MAGIC             ELSE date_add('day', -1, current_date)                                     -- Other days → yesterday only
# MAGIC         END AS start_date,
# MAGIC         date_add('day', -1, current_date) AS end_date
# MAGIC ),
# MAGIC welcome_calls AS (
# MAGIC     SELECT
# MAGIC         t.*,
# MAGIC         json_parse(t.meta_data) AS call_meta,
# MAGIC         try(date_parse(
# MAGIC             json_extract_scalar(json_parse(t.meta_data), '$.callInitiationTime'),
# MAGIC             '%Y/%m/%d %H:%i:%s +0530'
# MAGIC         )) AS call_timestamp
# MAGIC     FROM awsdatacatalog.crm_tickets.tickets t
# MAGIC     WHERE t.ticket_group = 'GI_WELCOME_CALLING'
# MAGIC       AND json_extract_scalar(json_parse(t.meta_data), '$.userId') = 'gousiya.gb@navi.com'
# MAGIC )
# MAGIC SELECT
# MAGIC     json_extract_scalar(call_meta, '$.callId') AS "Call ID",
# MAGIC     call_timestamp AS "Call Time",
# MAGIC     CAST(call_timestamp AS date) AS "Call Date",
# MAGIC     ticket_group AS "Process Name",
# MAGIC     title AS "Campaign Name",
# MAGIC     json_extract_scalar(call_meta, '$.leadId') AS "Lead ID",
# MAGIC     NULL AS "Lead Name",
# MAGIC     json_extract_scalar(call_meta, '$.customerPhoneNumber') AS "Phone",
# MAGIC     json_extract_scalar(call_meta, '$.policy_id') AS "Customer ID",
# MAGIC     source_channel AS "Call Type",
# MAGIC     json_extract_scalar(call_meta, '$.disposition') AS "System Disposition",
# MAGIC     NULL AS "Hangup Cause Code",
# MAGIC     json_extract_scalar(call_meta, '$.hangupDetails') AS "Hangup Details",
# MAGIC     json_extract_scalar(call_meta, '$.setupTime') AS "Customer Setup Time",
# MAGIC     json_extract_scalar(call_meta, '$.ringingTime') AS "Customer Ringing Time",
# MAGIC     json_extract_scalar(call_meta, '$.ivrTime') AS "IVR Time",
# MAGIC     json_extract_scalar(call_meta, '$.talkTime') AS "Customer Talk Time",
# MAGIC     json_extract_scalar(call_meta, '$.holdTime') AS "Customer Hold Duration",
# MAGIC     source_channel AS "Actual Channel",
# MAGIC     json_extract_scalar(call_meta, '$.numAttempts') AS "Num Attempts",
# MAGIC     NULL AS "Association Type",
# MAGIC     json_extract_scalar(call_meta, '$.userId') AS "User ID",
# MAGIC     split_part(json_extract_scalar(call_meta, '$.userId'), '@', 1) AS "User Name",
# MAGIC     json_extract_scalar(call_meta, '$.subDisposition') AS "Disposition Code",
# MAGIC     json_extract_scalar(call_meta, '$.disposition') AS "Disposition Class",
# MAGIC     NULL AS "Transfer To Agent/Phone",
# MAGIC     NULL AS "User Setup Time",
# MAGIC     json_extract_scalar(call_meta, '$.ringingTime') AS "User Ringing Time",
# MAGIC     json_extract_scalar(call_meta, '$.talkTime') AS "User Talk Time",
# MAGIC     NULL AS "ACW Duration",
# MAGIC     description AS "Call Notes",
# MAGIC     json_extract_scalar(call_meta, '$.unique_id') AS "Unique ID"
# MAGIC FROM welcome_calls
# MAGIC CROSS JOIN parameters
# MAGIC WHERE CAST(call_timestamp AS date) BETWEEN start_date AND end_date
# MAGIC ORDER BY call_timestamp DESC
# MAGIC """
# MAGIC
# MAGIC df  = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1XBMQrVQhWW4ZUYyqpW25kqJimnsZInj2AOuljKqrxSM/edit?gid=1204962317#gid=1204962317"
worksheet_name = "Master Dump"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)